# CephaloStudio AI audit — Phase 0/1/2 (multi-dataset)

Runs the evidence-gathering phase of
`docs/CephaloStudio_AI_Landmark_Improvement_Plan.md`:

- **Phase 0** — freeze the baseline model manifest.
- **Phase 1** — profile the datasets (Aariz + ISBI 2015).
- **Phase 2** — benchmark the **Aariz 29-landmark model on ISBI 2015**, scoring the
  19 shared landmarks against ISBI ground truth (the model never saw ISBI).

**Kaggle setup**
1. Accelerator: **None (CPU)** — ONNX inference only.
2. Internet: **On**.
3. *Recommended:* Add Data → **`jiahongqian/cephalometric-landmarks`** (ISBI 2015).
   Optionally also attach an Aariz dataset (for the baseline + domain comparison).
   If ISBI isn't attached, the notebook fetches it from the public mirror.

Outputs land in `/kaggle/working/audit/` (download the folder).

In [ ]:
!pip -q install onnxruntime scipy pillow numpy requests pandas
print("deps installed")

In [ ]:
import os
%cd /kaggle/working
!rm -rf CephalometryStudio
!git clone --depth 1 https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git
assert os.path.isdir("CephalometryStudio/scripts/ai"), "clone missing scripts/ai — push first"
print("cloned")

In [ ]:
import glob, os, urllib.request

# ISBI annotations (public mirror; matches ISBI19 order). Test + train splits.
os.makedirs("/kaggle/working/audit", exist_ok=True)
ANN = "/kaggle/working/audit/isbi_annotations_test.csv"
ANN_TRAIN = "/kaggle/working/audit/isbi_annotations_train.csv"
BASE = "https://raw.githubusercontent.com/stolariks/medical-landmark-detection/main/data/isbi-2015/"
if not os.path.exists(ANN):
    urllib.request.urlretrieve(BASE + "test/annotations.csv", ANN)
if not os.path.exists(ANN_TRAIN):
    urllib.request.urlretrieve(BASE + "train/annotations.csv", ANN_TRAIN)

stems = set()
for p in (ANN, ANN_TRAIN):
    with open(p) as f:
        for line in f.readlines()[1:]:
            if line.strip():
                stems.add(os.path.splitext(os.path.basename(line.split(",")[0]))[0])
print("ISBI annotation stems:", len(stems))

# Find ISBI images under /kaggle/input (any dir holding images named like the stems).
ISBI_DIR = None
for ext in ("*.jpg", "*.png", "*.jpeg"):
    for p in glob.glob("/kaggle/input/**/" + ext, recursive=True):
        if os.path.splitext(os.path.basename(p))[0] in stems:
            ISBI_DIR = os.path.dirname(p)
            break
    if ISBI_DIR:
        break
print("ISBI image dir:", ISBI_DIR or "(not attached — will fetch from mirror)")

# Optionally locate an attached Aariz split (for the baseline + domain comparison).
AARIZ_VALID = None
for base, dirs, _ in os.walk("/kaggle/input"):
    if os.path.basename(base) == "valid" and "Cephalograms" in dirs:
        AARIZ_VALID = base
        break
print("Aariz valid dir:", AARIZ_VALID or "(not attached)")

In [ ]:
import hashlib, os, requests

URL = "https://huggingface.co/MuhammadNabeelSh/cephalometry-landmarks/resolve/main/landmarks-cepha29.int8.onnx"
SHA = "f7f1db9cc2de37c0fa94dab431d7d3446d1d8ffc62fa92e6610c2380af83080d"
MODEL = "/kaggle/working/landmarks-cepha29.int8.onnx"
if not os.path.exists(MODEL):
    with requests.get(URL, stream=True, headers={"User-Agent": "Mozilla/5.0"}) as r:
        r.raise_for_status()
        with open(MODEL, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
h = hashlib.sha256()
with open(MODEL, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print("model MB:", round(os.path.getsize(MODEL) / 1e6, 1), "sha256:", h.hexdigest())
assert h.hexdigest() == SHA, "model sha256 mismatch"

In [ ]:
import subprocess, sys
# Phase 0 — freeze the baseline manifest (thresholds read from source).
subprocess.run([sys.executable, "CephalometryStudio/scripts/ai/baseline_freeze.py",
                "--model", MODEL, "--out", "CephalometryStudio/scripts/ai/baseline"],
               cwd="/kaggle/working", check=True)

In [ ]:
import subprocess, sys, os
# Phase 1 — profile datasets.
P = "CephalometryStudio/scripts/ai/profile_dataset.py"
if ISBI_DIR:
    subprocess.run([sys.executable, P, "--images", ISBI_DIR, "--name", "isbi2015",
                    "--ann-dir", os.path.dirname(ANN), "--limit", "1000",
                    "--out", "CephalometryStudio/scripts/ai/datasets"],
                   cwd="/kaggle/working", check=True)
if AARIZ_VALID:
    ann_valid = os.path.join(AARIZ_VALID, "Annotations", "Cephalometric Landmarks", "Senior Orthodontists")
    subprocess.run([sys.executable, P, "--images", os.path.join(AARIZ_VALID, "Cephalograms"),
                    "--name", "aariz", "--ann-dir", ann_valid, "--limit", "200",
                    "--out", "CephalometryStudio/scripts/ai/datasets"],
                   cwd="/kaggle/working", check=True)

In [ ]:
import subprocess, sys, os
# Aariz baseline metrics (only if Aariz valid is attached) — used for domain comparison.
AARIZ_METRICS = None
if AARIZ_VALID:
    AARIZ_METRICS = "/kaggle/working/audit/aariz_valid_metrics.json"
    ann_valid = os.path.join(AARIZ_VALID, "Annotations", "Cephalometric Landmarks", "Senior Orthodontists")
    subprocess.run([sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
                    "--model", MODEL, "--set", "cepha29", "--limit", "200", "--dark",
                    "--ann-dir", ann_valid, "--images", os.path.join(AARIZ_VALID, "Cephalograms"),
                    "--ssm", "CephalometryStudio/src/data/shapeModel.cepha29.json",
                    "--metrics-out", AARIZ_METRICS,
                    "--report-ood", "--ood-out", "/kaggle/working/audit/aariz_valid_ood.csv"],
                   cwd="/kaggle/working", check=True)
else:
    print("Aariz not attached — skipping baseline metrics; domain comparison will be limited.")

In [ ]:
import subprocess, sys, os
# Phase 2 — Aariz 29-landmark model on ISBI (19 shared landmarks).
# Uses the production orientation/polarity search + both ISBI annotation splits.
# Run 1 = SSM-refined (app behaviour); Run 2 = raw (no SSM).
base = [sys.executable, "CephalometryStudio/scripts/ai/external_benchmark.py",
        "--model", MODEL, "--limit", "400", "--search",
        "--ann-extra", ANN_TRAIN, "--overlay", "12"]
loc = ["--isbi-images", ISBI_DIR, "--isbi-ann", ANN] if ISBI_DIR else ["--fetch-isbi"]
cmp = ["--aariz-metrics", AARIZ_METRICS] if AARIZ_METRICS else []

run1 = base + ["--ssm", "CephalometryStudio/src/data/shapeModel.cepha29.json"] + loc + cmp + ["--out", "/kaggle/working/audit/E0_isbi"]
run2 = base + ["--no-ssm"] + loc + cmp + ["--out", "/kaggle/working/audit/E0b_isbi_raw"]
for cmd in (run1, run2):
    print(" ".join(cmd), "\n")
    subprocess.run(cmd, cwd="/kaggle/working", check=True)

In [ ]:
import glob, os, shutil
out = "/kaggle/working/audit"
os.makedirs(out, exist_ok=True)
# Collect the generated reports.
for src in ["CephalometryStudio/scripts/ai/baseline/model_manifest.json",
            "CephalometryStudio/scripts/ai/baseline/README.md",
            "CephalometryStudio/scripts/ai/datasets/registry.json",
            "CephalometryStudio/scripts/ai/datasets/dataset_statistics.csv"]:
    if os.path.exists(src):
        shutil.copy(src, out)
print("=== audit outputs (/kaggle/working/audit) ===")
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    if os.path.isfile(p):
        print(f"  {os.path.getsize(p):>10}  {f}")

## Next

Download `/kaggle/working/audit/`. The important file is
`E0_isbi/isbi_metrics.json` (and `failure_taxonomy.md`, `domain_comparison.csv`).

That report tells us **which landmarks and how badly** the Aariz model fails on
ISBI — i.e. the size of the domain shift. Only after reading it do we decide the
multi-domain training plan (E1–E4) and the landmark set (shared 19 vs masked 29).